# 05 — Kernel Density Estimation / 核密度估计

**Chapter 10 — File 5 of 5**

## Summary

We use Kernel Density Estimation (KDE) from sklearn to estimate the density of bimodal data. KDE is a non-parametric method that can capture complex multimodal distributions without assuming a specific form.

我们使用sklearn的核密度估计（KDE）来估计双峰数据的密度。KDE是一种非参数方法，可以捕获复杂的多峰分布，无需假设特定形式。

**Formula:**

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$

where $K$ is the kernel and $h$ is the bandwidth

---
## Background / 背景导读

**本文件主要内容 / What this file covers:**

- 训练模型 / Train the model
- 评估模型效果 / Evaluate model performance
- 可视化结果 / Visualize results

## Code Flow / 代码流程

```
   
┌──────────────────────┐
│  训练模型 Train Model  │
└──────────────────────┘
  │
  ▼
┌───────────────────┐
│  可视化 Visualize  │
└───────────────────┘
```

## Step 1 — Import Libraries / 导入库

In [ ]:
# 导入NumPy数值计算库 / Import NumPy numerical computing library
import numpy as np
# 导入Matplotlib绑图库 / Import Matplotlib plotting library
import matplotlib.pyplot as plt
from scipy.stats import norm
# 导入Scikit-learn机器学习库 / Import Scikit-learn ML library
from sklearn.neighbors import KernelDensity

## Step 2 — Generate Bimodal Data / 生成双峰数据

In [ ]:
# Generate bimodal data / 生成双峰数据
# 生成随机数 / Generate random numbers
np.random.seed(42)
# 生成随机数 / Generate random numbers
samples1 = np.random.normal(loc=20, scale=5, size=500)
# 生成随机数 / Generate random numbers
samples2 = np.random.normal(loc=40, scale=5, size=500)
# 拼接数组 / Concatenate arrays
bimodal_samples = np.concatenate([samples1, samples2])
bimodal_samples = bimodal_samples.reshape(-1, 1)  # Reshape for sklearn / 为sklearn重新整形

# 打印输出 / Print output
print(f'Kernel Density Estimation / 核密度估计')
# 查看数据形状（行数, 列数） / Check data shape (rows, columns)
print(f'Sample shape: {bimodal_samples.shape}')
# 打印输出 / Print output
print(f'Data range: [{bimodal_samples.min():.2f}, {bimodal_samples.max():.2f}]')

## Step 3 — Fit KDE / 拟合KDE

In [ ]:
# Fit KDE with different bandwidth values / 用不同的带宽值拟合KDE
bandwidths = [0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# 展平为一维数组 / Flatten to 1D array
axes = axes.flatten()

# 改变数组形状（不改变数据） / Reshape array (data unchanged)
x = np.linspace(0, 60, 1000).reshape(-1, 1)

# 同时获取索引和值 / Get both index and value
for idx, bw in enumerate(bandwidths):
    # Fit KDE / 拟合KDE
    kde = KernelDensity(bandwidth=bw, kernel='gaussian')
    kde.fit(bimodal_samples)
    
    # Score samples (log density) / 对样本评分（对数密度）
    log_density = kde.score_samples(x)
    density = np.exp(log_density)
    
    # Plot / 绘制
    ax = axes[idx]
    ax.hist(bimodal_samples, bins=30, density=True, alpha=0.5, 
            color='steelblue', edgecolor='black', label='Data')
    ax.plot(x, density, 'r-', linewidth=2, label='KDE')
    
    ax.set_xlabel('Value / 值')
    ax.set_ylabel('Density / 密度')
    ax.set_title(f'KDE with Bandwidth = {bw}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
# 显示图表 / Display the plot
plt.show()

## Step 4 — Compare Methods / 比较方法

In [ ]:
# Compare parametric, histogram, and KDE / 比较参数、直方图和KDE方法
fig, ax = plt.subplots(figsize=(12, 6))

# Data histogram / 数据直方图
ax.hist(bimodal_samples, bins=30, density=True, alpha=0.5, 
        color='steelblue', edgecolor='black', label='Data')

# True bimodal distribution / 真实双峰分布
# 生成等间距数组 / Generate evenly spaced array
x = np.linspace(0, 60, 1000)
true_bimodal = (norm.pdf(x, loc=20, scale=5) + norm.pdf(x, loc=40, scale=5)) / 2
ax.plot(x, true_bimodal, 'g-', linewidth=2.5, label='True Distribution')

# Parametric (single normal) / 参数方法（单一正态）
mu = bimodal_samples.mean()
sigma = bimodal_samples.std()
ax.plot(x, norm.pdf(x, loc=mu, scale=sigma), 'orange', linestyle='--', 
        linewidth=2, label='Parametric (Single Normal)')

# KDE / KDE
kde = KernelDensity(bandwidth=1.5, kernel='gaussian')
kde.fit(bimodal_samples)
# 改变数组形状（不改变数据） / Reshape array (data unchanged)
kde_density = np.exp(kde.score_samples(x.reshape(-1, 1)))
ax.plot(x, kde_density, 'r-', linewidth=2.5, label='KDE (Bandwidth=1.5)')

ax.set_xlabel('Value / 值')
ax.set_ylabel('Density / 密度')
ax.set_title('Comparison: Parametric vs Histogram vs KDE')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
# 显示图表 / Display the plot
plt.show()

## Learning Notes / 学习笔记

- **Concept**: KDE is a non-parametric density estimation method that places a kernel (typically Gaussian) at each data point and sums them. The bandwidth controls the smoothness: smaller bandwidth captures more detail but adds noise, while larger bandwidth smooths out details.
  
  **概念**：KDE是一种非参数密度估计方法，在每个数据点处放置一个核（通常是高斯的）并求和。带宽控制平滑度：较小的带宽捕获更多细节但增加噪声，较大的带宽平滑细节。

- **ML Application**: KDE is used for probability density estimation, anomaly detection (identifying low-density regions), and as a preprocessing step in machine learning pipelines for exploratory analysis.
  
  **机器学习应用**：KDE用于概率密度估计、异常检测（识别低密度区域）和作为机器学习管道中用于探索性分析的预处理步骤。

### Glossary / 术语速查

| 术语 Term | 中文解释 | English |
|-----------|---------|---------|
| `Flatten` | 展平多维为一维 | Flatten multi-dim to 1D |
| `matplotlib` | 绑图库 | Plotting library |
| `np.random` | 随机数生成 | Random number generation |
| `numpy` | 数值计算库 | Numerical computing library |
| `plt.show` | 显示图表 | Display plot |
| `plt.subplot` | 创建子图 | Create subplot |

## Chapter 10 Complete / 第10章完成

This chapter covered density estimation techniques from simple histograms to advanced KDE methods.

## Complete Code / 完整代码一览

In [ ]:
# Complete KDE Analysis / 完整KDE分析

# 导入NumPy数值计算库 / Import NumPy numerical computing library
import numpy as np
# 导入Matplotlib绑图库 / Import Matplotlib plotting library
import matplotlib.pyplot as plt
from scipy.stats import norm
# 导入Scikit-learn机器学习库 / Import Scikit-learn ML library
from sklearn.neighbors import KernelDensity

# Generate bimodal data / 生成双峰数据
# 生成随机数 / Generate random numbers
np.random.seed(42)
# 生成随机数 / Generate random numbers
samples1 = np.random.normal(loc=20, scale=5, size=500)
# 生成随机数 / Generate random numbers
samples2 = np.random.normal(loc=40, scale=5, size=500)
# 拼接数组 / Concatenate arrays
bimodal_samples = np.concatenate([samples1, samples2])
# 改变数组形状（不改变数据） / Reshape array (data unchanged)
bimodal_samples_reshaped = bimodal_samples.reshape(-1, 1)

# Fit KDE with different bandwidths / 用不同带宽拟合KDE
bandwidths = [0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# 展平为一维数组 / Flatten to 1D array
axes = axes.flatten()

# 改变数组形状（不改变数据） / Reshape array (data unchanged)
x = np.linspace(0, 60, 1000).reshape(-1, 1)

# 同时获取索引和值 / Get both index and value
for idx, bw in enumerate(bandwidths):
    kde = KernelDensity(bandwidth=bw, kernel='gaussian')
    kde.fit(bimodal_samples_reshaped)
    log_density = kde.score_samples(x)
    density = np.exp(log_density)
    
    ax = axes[idx]
    ax.hist(bimodal_samples, bins=30, density=True, alpha=0.5, 
            color='steelblue', edgecolor='black', label='Data')
    ax.plot(x[:, 0], density, 'r-', linewidth=2, label='KDE')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.set_title(f'KDE with Bandwidth = {bw}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
# 显示图表 / Display the plot
plt.show()

# Compare methods / 比较方法
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(bimodal_samples, bins=30, density=True, alpha=0.5, 
        color='steelblue', edgecolor='black', label='Data')

# 生成等间距数组 / Generate evenly spaced array
x_1d = np.linspace(0, 60, 1000)
true_bimodal = (norm.pdf(x_1d, loc=20, scale=5) + norm.pdf(x_1d, loc=40, scale=5)) / 2
ax.plot(x_1d, true_bimodal, 'g-', linewidth=2.5, label='True Distribution')

mu = bimodal_samples.mean()
sigma = bimodal_samples.std()
ax.plot(x_1d, norm.pdf(x_1d, loc=mu, scale=sigma), 'orange', linestyle='--', 
        linewidth=2, label='Parametric')

kde = KernelDensity(bandwidth=1.5, kernel='gaussian')
kde.fit(bimodal_samples_reshaped)
# 改变数组形状（不改变数据） / Reshape array (data unchanged)
kde_density = np.exp(kde.score_samples(x_1d.reshape(-1, 1)))
ax.plot(x_1d, kde_density, 'r-', linewidth=2.5, label='KDE')

ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.set_title('Comparison: Parametric vs KDE')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
# 显示图表 / Display the plot
plt.show()